# Mutation count probabilities
So far, we have only looked at statistics based on branch lengths of the coalescent tree. However, when dealing with short sequences, we may not have enough mutations to compute stable branch-length-based summary statistics. Instead, we may like to obtain the distribution of mutational counts. It is particularly informative to consider the SFS computed over small non-recombining blocks.

In the following example we obtain the first 8 unfolded mutational configuration probabilities under a three-epoch size-change demography using a mutation rate of 1. {meth}`~phasegen.distributions.UnfoldedSFSDistribution.get_mutation_configs` returns a generator that yields the mutational configurations in descending order of probability, so a target probability mass is reached after evaluating comparatively few of them. Each configuration is a vector of length `n-1` where the `i`th entry denotes the number of mutations with multiplicities `i+1`, and `n` is the number of lineages. For example, `[1, 1, 0]` means that there is one singleton, one doubleton, and no tripleton mutations.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
import phasegen as pg
import pandas as pd
from itertools import islice

# a three-epoch size-change demography (decline then expansion)
coal = pg.Coalescent(n=4, demography=pg.Demography(pop_sizes={'pop_0': {0: 1, 0.5: 0.3, 1.5: 2}}))

pd.DataFrame(islice(coal.sfs.get_mutation_configs(theta=1), 8))
```

```{glue} python-s0-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
library(phasegen)
pg <- load_phasegen()

# a three-epoch size-change demography (decline then expansion)
coal <- pg$Coalescent(
    n = 4L,
    demography = pg$Demography(
        pop_sizes = list(pop_0 = 1),
        events = c(
            pg$PopSizeChange(pop = "pop_0", time = 0.5, size = 0.3),
            pg$PopSizeChange(pop = "pop_0", time = 1.5, size = 2)
        )
    )
)

do.call(rbind, reticulate::iterate(
    pg$take_n(coal$sfs$get_mutation_configs(theta = 1), 8L)
))
```

```{glue} r-s0-0-0
```

~~~~~

~~~~~~


The number of mutational configurations is infinite since we may have arbitrarily many mutations, albeit with increasingly lower probabilities depending on the mutation rate and coalescent distribution. We may also wish to obtain probabilities until we have reached a certain probability mass threshold. We can do this by consuming the generator while keeping track of the {attr}`~phasegen.distributions.UnfoldedSFSDistribution.generated_mass` attribute. In the following example, we obtain mutational configurations until the generated mass is above 0.8.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
it = coal.sfs.get_mutation_configs(theta=1)

# continue until generated mass is above 0.8
pd.DataFrame(pg.takewhile_inclusive(lambda _: coal.sfs.generated_mass < 0.8, it))
```

```{glue} python-s1-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
it <- coal$sfs$get_mutation_configs(theta = 1)

until <- pg$takewhile_inclusive(function(x) {
    return(coal$sfs$generated_mass < 0.8)
}, it)

# continue until generated mass is above 0.8
do.call(rbind, reticulate::iterate(until))
```

```{glue} r-s1-0-0
```

~~~~~

~~~~~~


Alternatively, we can obtain the probability of a specific mutational configuration (cf. {meth}`~phasegen.distributions.UnfoldedSFSDistribution.get_mutation_config`).

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
coal.sfs.get_mutation_config([1, 1, 0], theta=1)
```

```{glue} python-s2-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
coal$sfs$get_mutation_config(c(1, 1, 0), theta = 1)
```

```{glue} r-s2-0-0
```

~~~~~

~~~~~~


We can also do the same for folded configurations. In this case, the configurations are vectors of length `n // 2`. For example, configuration `[1, 1]` denotes one singleton or tripleton and one doubleton mutation.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
df = pd.DataFrame(islice(coal.fsfs.get_mutation_configs(theta=1), 30))
      
df.plot(kind='bar', x=0, legend=False, xlabel='config');
```

```{glue} python-s3-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
df <- do.call(rbind, reticulate::iterate(
    pg$take_n(coal$fsfs$get_mutation_configs(theta = 1), 30L)
))

heights <- as.numeric(df[, 2])
labels <- sapply(df[, 1], function(x) paste(unlist(x), collapse = ", "))

barplot(heights, names.arg = labels, las = 2, xlab = "config", cex.names = 0.6)
```

```{glue} r-s3-0-0
```

~~~~~

~~~~~~
